# Lab 03: Quick Commerce Dataset

Guillermo Alejandro Romero Vázquez

In [60]:
from SparkUtils import SparkUtils

from pyspark.sql.functions import lit, col, when, count, avg

In [61]:
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Lab 03"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

In [62]:
column_types = [
    ("Order_ID", "long"),
    ("Company", "string"),
    ("City", "string"),
    ("Customer_Age", "int"),
    ("Order_Value", "float"),
    ("Delivery_Time_Min", "float"),
    ("Distance_Km", "float"),
    ("Items_Count", "float"),
    ("Product_Category", "string"),
    ("Payment_Method", "string"),
    ("Customer_Rating", "float"),
    ("Discount_Applied", "float"),
    ("Delivery_Partner_Rating", "float"),
]

qcommerce_schema = SparkUtils.generate_schema(column_types)

qcommerce_df = spark \
    .read \
    .option("header", "true") \
    .schema(qcommerce_schema) \
    .csv("/opt/spark/work-dir/data/quick_commerce/")

qcommerce_df.printSchema()

qcommerce_df.show(5)

print(f"Rows: {qcommerce_df.count()}")

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+----

In [63]:
qcommerce_null_count_df = SparkUtils.count_nulls(qcommerce_df)

qcommerce_null_count_df.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [64]:
no_nulls_qcommerce_df = qcommerce_df.dropna()

print(f"Rows: {no_nulls_qcommerce_df.count()}")

Rows: 780992


In [65]:
fillna_qcommerce_df = qcommerce_df.fillna({
    "City": "Unknown",
    "Items_Count": 0.0,
    "Customer_Rating": 0.0,
    "Delivery_Partner_Rating": 0.0
})

print(f"Rows: {fillna_qcommerce_df.count()}")

Rows: 1000000


In [66]:
qcommerce_df = fillna_qcommerce_df.withColumn("Country_Code", lit("IN"))

qcommerce_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Country_Code|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [67]:
qcommerce_df = qcommerce_df.withColumn("Paid_Tax", col("Order_Value") * 0.16)

qcommerce_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Country_Code|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

### Task 1: Delivery SLA Classification + Filtering
Create Delivery_SLA using withColumn + when:

- Delivery_Time_Min ≤ 15 → “FAST”,  15–20 → “ON_TIME”, and  > 20 → “LATE”
- Filter only Delivery_SLA = “LATE” and orderBy Delivery_Time_Min (descending).
- Display: Order_ID, Company, City, Delivery_Time_Min, Delivery_SLA.

In [68]:
qcommerce_df = qcommerce_df\
    .withColumn(\
        "Delivery_SLA",\
        when(col("Delivery_Time_Min") <= 15, "FAST")\
        .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME")\
        .when(col("Delivery_Time_Min") > 20, "LATE")\
    ).filter(col("Delivery_SLA") == "LATE")\
    .orderBy(col("Delivery_Time_Min"), ascending=False)

qcommerce_df\
    .select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA")\
    .show(5)

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
+--------+--------+--------+-----------------+------------+
only showing top 5 rows


### Task 2: Discount impact (effective price + bucket + grouping)
- Create Effective_Order_Value = Order_Value * (1 - Discount_Applied).
- Create Price_Bucket using when:
    - < 200 → “LOW”
    - 200–500 → “MEDIUM”
    - 500 → “HIGH”
- Group by Price_Bucket and compute:
    - count(*)
    - avg(Effective_Order_Value)
- Order by average effective value (descending).

In [69]:
qcommerce_df = qcommerce_df.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied")))

agg_qcommerce_df = qcommerce_df\
    .withColumn(
        "Price_Bucket",\
        when(col("Order_Value") < 200, "LOW")\
        .when((col("Order_Value") >= 200) & (col("Order_Value") < 500), "MEDIUM")\
        .when(col("Order_Value") >= 500, "HIGH")
    ).groupBy("Price_Bucket")\
    .agg(
        count("*").alias("Count_Price_Bucket"),
        avg("Effective_Order_Value").alias("Avg_Effective_Order_Value")
    ).orderBy("Avg_Effective_Order_Value", ascending=False)

agg_qcommerce_df.show(10)

+------------+------------------+-------------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Order_Value|
+------------+------------------+-------------------------+
|        HIGH|            116784|       339.20318478658635|
|      MEDIUM|             83778|       252.42027661844153|
|         LOW|             59086|       61.736481132090255|
+------------+------------------+-------------------------+



In [70]:
qcommerce_df = qcommerce_df\
    .withColumn(
        "Age_Group",
        when(col("Customer_Age") <= 25, "YOUNG")\
        .when((col("Customer_Age") > 25) & (col("Customer_Age") <= 44), "ADULT")\
        .when(col("Customer_Age") >= 45, "SENIOR")
    ).filter((col("Customer_Age") >= 0) & (col("Customer_Age") <= 100))\
    .groupBy("Age_Group", "Product_Category")\
    .agg(
        count("*").alias("orders"),
        avg("Order_Value").alias("Avg_Order_Value")
    ).orderBy(col("Age_Group").asc())\
    .orderBy(col("orders").desc())

qcommerce_df.show(5)

+---------+-------------------+------+------------------+
|Age_Group|   Product_Category|orders|   Avg_Order_Value|
+---------+-------------------+------+------------------+
|    ADULT|      Personal Care| 17078|495.25267501591605|
|    ADULT|              Dairy| 16901|  502.789107149631|
|    ADULT|          Household| 16863| 497.2638757191384|
|    ADULT|Fruits & Vegetables| 16823| 494.2270302383921|
|    ADULT|             Snacks| 16809| 499.4955506958792|
+---------+-------------------+------+------------------+
only showing top 5 rows


In [72]:
spark.stop()